# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`: Arsalaan Alam**  
**`Roll Number`: U20340064**  
**`GitHub Branch`:** arsalaan_U20230064

# Imports and Setup

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# from rlcmab_sampler import sampler


# Load Datasets

In [3]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [4]:
# Data Preprocessing for User Classification

# Check missing values
print("Missing values in train_users:")
print(train_users.isnull().sum())
print(f"\nTotal rows: {len(train_users)}")

# Check unique values in label column
print(f"\nUser categories (labels): {train_users['label'].unique()}")

Missing values in train_users:
user_id                          0
age                            698
income                           0
clicks                           0
purchase_amount                  0
session_duration                 0
content_variety                  0
engagement_score                 0
num_transactions                 0
avg_monthly_spend                0
avg_cart_value                   0
browsing_depth                   0
revisit_rate                     0
scroll_activity                  0
time_on_site                     0
interaction_count                0
preferred_price_range            0
discount_usage_rate              0
wishlist_size                    0
product_views                    0
repeat_purchase_gap (days)       0
churn_risk_score                 0
loyalty_index                    0
screen_brightness                0
battery_percentage               0
cart_abandonment_count           0
browser_version                  0
background_app_count    

## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [5]:

df = train_users.copy()
df = df.drop(columns=['user_id'])
df['age'] = df['age'].fillna(df['age'].median())

# Encode categorical features
label_encoders = {}

# Encode region_code
le_region = LabelEncoder()
df['region_code'] = le_region.fit_transform(df['region_code'])
label_encoders['region_code'] = le_region

# Encode browser_version
le_browser = LabelEncoder()
df['browser_version'] = le_browser.fit_transform(df['browser_version'])
label_encoders['browser_version'] = le_browser

# Encode subscriber (boolean to int)
df['subscriber'] = df['subscriber'].astype(int)

# Separate features and target
X = df.drop(columns=['label'])
y = df['label']

# Encode target labels
le_label = LabelEncoder()
y_encoded = le_label.fit_transform(y)
label_encoders['label'] = le_label

print(f"Features shape: {X.shape}")
print(f"Target classes: {le_label.classes_}")

Features shape: (2000, 31)
Target classes: ['user_1' 'user_2' 'user_3']


In [6]:
# Split into training (80%) and validation (20%) sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

Training set size: 1600
Validation set size: 400


In [7]:
# Scale features for better performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Use Voting Ensemble (combines multiple classifiers)
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression

print("Training Voting Ensemble Classifier...")

# Create ensemble of multiple classifiers
rf = RandomForestClassifier(n_estimators=500, max_depth=None, random_state=42, n_jobs=-1)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)

clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('lr', lr)],
    voting='soft',
    n_jobs=-1
)
clf.fit(X_train_scaled, y_train)

# Predict on validation set
y_pred = clf.predict(X_val_scaled)

# Evaluate the classifier
print("\nClassification Report on Validation Set:\n")
print(classification_report(y_val, y_pred, target_names=le_label.classes_))

# Also print accuracy
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.4f}")

Training Voting Ensemble Classifier...

Classification Report on Validation Set:

              precision    recall  f1-score   support

      user_1       0.89      0.87      0.88       142
      user_2       1.00      0.88      0.94       142
      user_3       0.84      0.99      0.91       116

    accuracy                           0.91       400
   macro avg       0.91      0.91      0.91       400
weighted avg       0.91      0.91      0.91       400

Validation Accuracy: 0.9075


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
